# Notebook 03 — CaFE Locality Analysis
**Owner: Person B (CaFE IG check, ERF visualisation) + Person C (Spearman correlation, layer evolution) — Week 2**

Stage 3. Test the locality hypothesis: do monosemantic SAE features in DINO ViT-B/16 respond
to locally bounded spatial regions, or do they integrate over the whole image?

**Method:**
- Select top-N features per layer (layers 4, 6, 9) ranked by Monosemanticity Score from notebook 02
- Run CaFE Integrated Gradients check: compare max-activation patch location vs. max-IG attribution location
- Compute Spearman correlation between MS score and CaFE agreement rate

**Requires:** notebook 01 (cache) and notebook 02 (top patches, MS scores, CLIP labels) completed.

## 1. Load activations and feature outputs from notebook 02

In [ ]:
"""Environment — run this cell first."""
import gzip, pickle, json, math
from pathlib import Path

import torch
from src.config import get_config
import src.config as _cfg_mod

cfg       = get_config()
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")
if device == "cuda":
    free  = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  {free:.1f}/{total:.1f} GB free")

In [ ]:
"""Step 1 — load activations, image index, top patches, and MS scores from notebook 02 outputs.

All data is read from disk — no model forward pass in this cell.
Run notebooks/01_sae_setup.ipynb and notebooks/02_feature_analysis.ipynb first.
"""
from src.cache import load_layer, load_image_index, load_metadata

cache_path      = repo_root / cfg.outputs.cache_path
feature_out_dir = repo_root / "outputs" / "features"

meta  = load_metadata(cachepath=str(cache_path))
index = load_image_index(cachepath=str(cache_path))
image_paths = index["paths"]

print(f"Model  : {meta['model_name']}")
print(f"Layers : {meta['layers']}")
print(f"Images : {meta['n_images']}")

# Residual-stream activations per target layer
activations_by_layer = {}
for layer in cfg.sae.target_layers:
    activations_by_layer[layer] = load_layer(layer, cachepath=str(cache_path))
    print(f"  layer {layer}: {tuple(activations_by_layer[layer].shape)}")

# Top patches per layer (notebook 02, cell 3)
all_top_patches_by_layer = {}
for layer in cfg.sae.target_layers:
    pkl_path = feature_out_dir / f"top_patches_layer{layer}_full.pkl.gz"
    if not pkl_path.exists():
        print(f"[WARNING] {pkl_path} not found — run notebook 02 cell 3 for layer {layer}")
        continue
    with gzip.open(pkl_path, "rb") as f:
        all_top_patches_by_layer[layer] = pickle.load(f)
    print(f"  top_patches layer {layer}: {len(all_top_patches_by_layer[layer]):,} features")

# MS scores per layer (notebook 02, cell 5)
ms_scores_by_layer = {}
MS_MAX_PATCHES = cfg.features.ms_max_patches
for layer in cfg.sae.target_layers:
    scores_path = feature_out_dir / f"ms_scores_layer{layer}_top{MS_MAX_PATCHES}.json"
    if not scores_path.exists():
        print(f"[WARNING] {scores_path} not found — run notebook 02 cell 5 for layer {layer}")
        continue
    with open(scores_path, encoding="utf-8") as f:
        ms_scores_by_layer[layer] = {int(k): v for k, v in json.load(f).items()}
    print(f"  ms_scores  layer {layer}: {len(ms_scores_by_layer[layer]):,} features")

print("\nAll data loaded.")

## 2. Feature selection

**Run 1** (index-based) — first 100 features by index per layer.  
Matches CaFE protocol exactly. Used for: headline Spearman ρ (full MS range), DINO vs. CLIP depth comparison (Fig 6 ★), locality-by-depth (Fig 4).

**Run 2** (MS-ranked) — top 50 by MS score per layer (support-filtered).  
Used for: category-stratified locality (Fig 5), range-restricted Spearman ρ.

In [ ]:
"""Step 2b — Run 2 feature selection: top 50 by MS score, support-filtered, per layer.

Selection criteria (identical to notebook 02 gallery filter):
  - MS score is not NaN
  - >= cfg.features.ms_max_patches patches with a valid patch_row
  - max activation >= cfg.features.min_activation_threshold

Features ranked by MS score descending; top cfg.cafe.n_images selected.
"""
N_CAFE    = cfg.cafe.n_images
MIN_PATCH = cfg.features.ms_max_patches
MIN_ACT   = cfg.features.min_activation_threshold

run2_features_by_layer = {}

for layer in cfg.sae.target_layers:
    if layer not in ms_scores_by_layer or layer not in all_top_patches_by_layer:
        print(f"[skip] layer {layer}: missing MS scores or top patches")
        continue

    eligible = []
    for fi, score in ms_scores_by_layer[layer].items():
        if score is None or math.isnan(score):
            continue
        patches = [p for p in all_top_patches_by_layer[layer].get(fi, [])
                   if p.get("patch_row") is not None]
        if len(patches) < MIN_PATCH:
            continue
        if max((p["activation_value"] for p in patches), default=0.0) < MIN_ACT:
            continue
        eligible.append((fi, score))

    eligible.sort(key=lambda kv: kv[1], reverse=True)
    selected = [fi for fi, _ in eligible[:N_CAFE]]
    run2_features_by_layer[layer] = selected

    print(f"Layer {layer}: Run 2 — {len(eligible):,} eligible → selected top {len(selected)} by MS")
    if selected:
        lo = ms_scores_by_layer[layer][selected[-1]]
        hi = ms_scores_by_layer[layer][selected[0]]
        print(f"  MS range: {lo:.3f} … {hi:.3f}")

# Keep backward-compat alias used in subsequent cells
selected_features_by_layer = run2_features_by_layer

In [ ]:
"""Step 2a — Run 1 feature selection: first 100 features by index, per layer.

CaFE protocol: select features[0:100] regardless of MS score.
This gives the full MS range needed for the headline Spearman ρ (symmetric with CLIP Run 3).
Also extracts MS scores for these 100 index-based features per layer (needed for D6 Spearman).
"""
RUN1_N = 100

run1_features_by_layer = {}
run1_ms_by_layer = {}

for layer in cfg.sae.target_layers:
    if layer not in ms_scores_by_layer:
        print(f"[skip] layer {layer}: MS scores not found")
        continue

    # First 100 feature indices (0-indexed, regardless of activation or MS)
    all_feature_ids = sorted(ms_scores_by_layer[layer].keys())
    selected = all_feature_ids[:RUN1_N]
    run1_features_by_layer[layer] = selected

    # Extract MS scores for these index-based features (NaN excluded for Spearman)
    run1_ms_by_layer[layer] = {
        fi: ms_scores_by_layer[layer][fi]
        for fi in selected
        if ms_scores_by_layer[layer].get(fi) is not None
           and not math.isnan(ms_scores_by_layer[layer][fi])
    }

    print(f"Layer {layer}: Run 1 — {len(selected)} features by index "
          f"| {len(run1_ms_by_layer[layer])} with valid MS score"
          f" (range {min(run1_ms_by_layer[layer].values()):.3f}"
          f"–{max(run1_ms_by_layer[layer].values()):.3f})")

## 3. CaFE IG — Run 1 index-based (Person B, D4/D5)

First 100 features by index per layer, `feature_selection="index"`.  
Results saved to `outputs/features/cafe_ig/run1/`.  
These results feed: Fig 4 (locality by depth), Fig 6 ★ (DINO vs. CLIP comparison), headline Spearman ρ.

In [ ]:
"""Step 5 — CaFE IG loop for Run 2 (MS-ranked, top 50 per layer).

Estimated time: ~5–15 s per feature on A5000 with ig_steps=50.
Results saved to outputs/features/cafe_ig/run2/.
erf_scores kept in-memory in cafe_run2_by_layer for ERF visualisation (cell 6).
"""
from src.model import get_model
from src.causal import cafe_sanity_check

if "model" not in dir():
    model = get_model()

cafe_run2_dir = repo_root / "outputs" / "features" / "cafe_ig" / "run2"
cafe_run2_dir.mkdir(parents=True, exist_ok=True)

cafe_run2_by_layer = {}

for layer in cfg.sae.target_layers:
    if layer not in run2_features_by_layer:
        continue

    selected    = run2_features_by_layer[layer]
    activations = activations_by_layer[layer]
    results_this_layer = {}

    print(f"\n── Layer {layer}  ({len(selected)} features, Run 2 MS-ranked) ──")
    for feat_idx in selected:
        result_path = cafe_run2_dir / f"cafe_ig_layer{layer}_feat{feat_idx}.json"

        if result_path.exists():
            with open(result_path, encoding="utf-8") as f:
                result = json.load(f)
            print(f"  [cached] feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}")
        else:
            result = cafe_sanity_check(
                layer, feat_idx, activations, image_paths, model,
                top_k=cfg.cafe.n_images,
                attribution_method="integrated_gradients",
                ig_steps=cfg.cafe.ig_steps,
                feature_selection="ms_ranked",
            )
            slim = {k: v for k, v in result.items() if k != "erf_scores"}
            with open(result_path, "w", encoding="utf-8") as f:
                json.dump(slim, f, indent=2)
            print(f"  feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}  (saved)")

        results_this_layer[feat_idx] = result

    cafe_run2_by_layer[layer] = results_this_layer
    ag = [r["agreement_rate"] for r in results_this_layer.values()]
    print(f"  mean agreement: {sum(ag)/len(ag):.3f}")

# Backward-compat alias used in ERF visualisation cell below
cafe_results_by_layer = cafe_run2_by_layer
print("\nRun 2 CaFE IG complete.")

In [ ]:
"""Step 3a — CaFE IG loop for Run 1 (index-based, first 100 features per layer).

TODO: Person B (D4 — layer 9 first; D5 — extend to layers 4 and 6).

Validate on 5 features before the full run (D4 requirement):
  IG maps must be spatially coherent (non-uniform). If uniform, debug
  the residual-skip detach in cafe_sanity_check before proceeding.

Expected runtime: ~5–15 s per feature on A5000 with ig_steps=50.
100 features × 3 layers × ~10 s ≈ ~50 min total.
"""
from src.model import get_model
from src.causal import cafe_sanity_check

model = get_model()

cafe_run1_dir = repo_root / "outputs" / "features" / "cafe_ig" / "run1"
cafe_run1_dir.mkdir(parents=True, exist_ok=True)

cafe_run1_by_layer = {}

for layer in cfg.sae.target_layers:
    if layer not in run1_features_by_layer:
        continue

    selected    = run1_features_by_layer[layer]
    activations = activations_by_layer[layer]
    results_this_layer = {}

    print(f"\n── Layer {layer}  ({len(selected)} features, Run 1 index-based) ──")
    for feat_idx in selected:
        result_path = cafe_run1_dir / f"cafe_ig_layer{layer}_feat{feat_idx}.json"

        if result_path.exists():
            with open(result_path, encoding="utf-8") as f:
                result = json.load(f)
            print(f"  [cached] feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}"
                  f"  top5={result.get('top5_agreement_rate', float('nan')):.2f}")
        else:
            result = cafe_sanity_check(
                layer, feat_idx, activations, image_paths, model,
                top_k=cfg.cafe.n_images,
                attribution_method="integrated_gradients",
                ig_steps=cfg.cafe.ig_steps,
                feature_selection="index",
            )
            slim = {k: v for k, v in result.items() if k != "erf_scores"}
            with open(result_path, "w", encoding="utf-8") as f:
                json.dump(slim, f, indent=2)
            print(f"  feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}"
                  f"  top5={result['top5_agreement_rate']:.2f}  (saved)")

        results_this_layer[feat_idx] = result

    cafe_run1_by_layer[layer] = results_this_layer
    ag = [r["agreement_rate"] for r in results_this_layer.values()]
    ag5 = [r.get("top5_agreement_rate", float("nan")) for r in results_this_layer.values()]
    print(f"  mean agreement: {sum(ag)/len(ag):.3f}  |  "
          f"mean top5: {sum(ag5)/len(ag5):.3f}")

print("\nRun 1 CaFE IG complete.")

## 6. ERF visualisation — Run 2 (Person B)

Saliency map + activation/IG-peak patch locations for a sample of Run 2 MS-ranked features.
Requires `erf_scores` in memory — cell 5 must have been run in this session (not loaded from cache).
Produces per-feature PNG → **report/figures/**.

In [ ]:
"""Step 4 — visualise ERF maps for a sample of features per layer (Person B)."""
from src.visualise import plot_cafe_comparison

figures_dir = repo_root / cfg.outputs.report_figures_path
figures_dir.mkdir(parents=True, exist_ok=True)

N_SHOW = 3  # features to display per layer

for layer in cfg.sae.target_layers:
    if layer not in cafe_results_by_layer:
        continue
    show = list(cafe_results_by_layer[layer].keys())[:N_SHOW]
    for feat_idx in show:
        result = cafe_results_by_layer[layer][feat_idx]
        if not result.get("erf_scores"):
            print(f"  [skip] layer {layer} feat {feat_idx}: no erf_scores in memory "
                  f"(loaded from cache — re-run cell 3 without cache to regenerate)")
            continue
        save_path = figures_dir / f"cafe_ig_layer{layer}_feat{feat_idx}.png"
        fig = plot_cafe_comparison(result, feat_idx, save_path=str(save_path))
        display(fig)
        print(f"  Saved → {save_path}")

## 7. CLIP-B/32 CaFE Run 3 — index-based (Person C, D6)

Reload config to CLIP, run `cafe_sanity_check` on first 100 features by index per layer.  
Same protocol as Run 1 (index-based). Results saved to `outputs/features/cafe_ig/run3_clip/`.  
Feeds: **Fig 6 ★** (DINO vs. CLIP comparison) and **CLIP Spearman ρ**.

**Prerequisites:** notebook 04 must be complete (CLIP cache + top patches + MS scores).

In [ ]:
"""Step 7a — Reload config to CLIP and load CLIP cache + MS scores for Run 3."""
from src.config import get_config
from src.cache import load_layer as _load_layer, load_image_index as _load_index
import gzip, pickle, json as _json

cfg_clip      = get_config("../configs/clip_b32.yaml", reload=True)
clip_cache    = repo_root / cfg_clip.outputs.cache_path
clip_feat_dir = repo_root / "outputs" / "features"

clip_index = _load_index(cachepath=str(clip_cache))
clip_image_paths = clip_index["paths"]

clip_activations_by_layer = {}
for _layer in cfg_clip.sae.target_layers:
    clip_activations_by_layer[_layer] = _load_layer(_layer, cachepath=str(clip_cache))
    print(f"  CLIP layer {_layer}: {tuple(clip_activations_by_layer[_layer].shape)}")

clip_ms_by_layer = {}
_MS_MAX = cfg_clip.features.ms_max_patches
for _layer in cfg_clip.sae.target_layers:
    _path = clip_feat_dir / f"ms_scores_clip_layer{_layer}_top{_MS_MAX}.json"
    if not _path.exists():
        print(f"[WARNING] {_path.name} not found — run notebook 04 first")
        continue
    with open(_path, encoding="utf-8") as _f:
        clip_ms_by_layer[_layer] = {int(k): v for k, v in _json.load(_f).items()}
    print(f"  CLIP ms_scores layer {_layer}: {len(clip_ms_by_layer[_layer]):,} features")

# Run 3 index-based selection (first 100 per layer, same as Run 1)
run3_features_by_layer = {}
run3_ms_by_layer = {}
for _layer in cfg_clip.sae.target_layers:
    if _layer not in clip_ms_by_layer:
        continue
    _all = sorted(clip_ms_by_layer[_layer].keys())[:RUN1_N]
    run3_features_by_layer[_layer] = _all
    run3_ms_by_layer[_layer] = {
        fi: clip_ms_by_layer[_layer][fi]
        for fi in _all
        if clip_ms_by_layer[_layer].get(fi) is not None
           and not math.isnan(clip_ms_by_layer[_layer][fi])
    }
    print(f"  Run 3 layer {_layer}: {len(_all)} features selected")

In [ ]:
"""Step 7b — CaFE IG loop for CLIP Run 3 (index-based, first 100 features per layer).

Uses CLIP model loaded via get_model() with cfg_clip active.
Results saved to outputs/features/cafe_ig/run3_clip/.
"""
from src.model import get_model as _get_model
from src.causal import cafe_sanity_check as _cafe_check

clip_model = _get_model(model_name=cfg_clip.model.name)

cafe_run3_dir = repo_root / "outputs" / "features" / "cafe_ig" / "run3_clip"
cafe_run3_dir.mkdir(parents=True, exist_ok=True)

cafe_run3_by_layer = {}

for layer in cfg_clip.sae.target_layers:
    if layer not in run3_features_by_layer:
        continue

    selected    = run3_features_by_layer[layer]
    activations = clip_activations_by_layer[layer]
    results_this_layer = {}

    print(f"\n── CLIP Layer {layer}  ({len(selected)} features, Run 3 index-based) ──")
    for feat_idx in selected:
        result_path = cafe_run3_dir / f"cafe_ig_layer{layer}_feat{feat_idx}.json"

        if result_path.exists():
            with open(result_path, encoding="utf-8") as f:
                result = json.load(f)
            print(f"  [cached] feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}")
        else:
            result = _cafe_check(
                layer, feat_idx, activations, clip_image_paths, clip_model,
                top_k=cfg_clip.cafe.n_images,
                attribution_method="integrated_gradients",
                ig_steps=cfg_clip.cafe.ig_steps,
                feature_selection="index",
            )
            slim = {k: v for k, v in result.items() if k != "erf_scores"}
            with open(result_path, "w", encoding="utf-8") as f:
                json.dump(slim, f, indent=2)
            print(f"  feat {feat_idx:6d}: agreement={result['agreement_rate']:.2f}  (saved)")

        results_this_layer[feat_idx] = result

    cafe_run3_by_layer[layer] = results_this_layer
    ag = [r["agreement_rate"] for r in results_this_layer.values()]
    print(f"  mean agreement: {sum(ag)/len(ag):.3f}")

print("\nRun 3 (CLIP) CaFE IG complete.")

# Restore DINO config for subsequent cells
cfg = get_config("../configs/default.yaml", reload=True)

## 8. Spearman correlation: MS score vs. CaFE agreement rate (Person C, D6)

Three variants required (§3 of the project plan):

| Variant | Sample | Pooling | Purpose |
|---------|--------|---------|---------|
| DINO Run 1 | 300 features, full MS range | all layers | Headline DINO ρ, symmetric with CLIP |
| DINO Run 2 | 150 features, MS-restricted | all layers | Range-restricted; noted as conservative |
| CLIP Run 3 | 300 features, full MS range | all layers | Headline CLIP ρ |

Per-layer ρ also reported for each variant. NaN MS features excluded; effective N documented.

In [ ]:
"""Step 8 — Spearman ρ: MS score vs. CaFE agreement rate — three variants (Person C, D6).

TODO: Person C — implement this cell from scratch.

Inputs available in memory:
  cafe_run1_by_layer  {layer: {feat_idx: result_dict}}  DINO Run 1, index-based
  cafe_run2_by_layer  {layer: {feat_idx: result_dict}}  DINO Run 2, MS-ranked
  cafe_run3_by_layer  {layer: {feat_idx: result_dict}}  CLIP Run 3, index-based
  run1_ms_by_layer    {layer: {feat_idx: ms_score}}     DINO Run 1 MS scores
  ms_scores_by_layer  {layer: {feat_idx: ms_score}}     DINO all features (for Run 2)
  run3_ms_by_layer    {layer: {feat_idx: ms_score}}     CLIP Run 3 MS scores

result_dict keys: 'agreement_rate', 'top5_agreement_rate', 'feature_selection', ...

Required output — spearman_results dict:
  {
    'dino_run1': {'per_layer': {4: {'rho':..,'pval':..,'n':..}, 6:.., 9:..},
                  'pooled_rho':..., 'pooled_pval':..., 'pooled_n':...},
    'dino_run2': { same structure },
    'clip_run3': { same structure },
  }

Steps:
  1. For each of the three variants, pair (ms_score, agreement_rate) per layer.
     Skip NaN MS features (dead features per Pach estimator); log effective N.
  2. Compute per-layer Spearman ρ and p-value using scipy.stats.spearmanr.
  3. Compute pooled ρ (concatenate all layers). Note in output: pooled ρ may
     reflect a shared depth trend — always report per-layer values alongside.
  4. Print a summary table: variant | layer | ρ | p | N.
  5. Call plot_ms_locality_scatter()    → save Fig 8 per model/variant.
  6. Call plot_locality_by_depth()      → save Fig 4 (DINO Run 1 agreement rates).
  7. Call plot_locality_comparison()    → save Fig 6 ★.
       - dino_rates from Run 1 agreement rates per layer.
       - clip_rates from Run 3 agreement rates per layer.
       - cafe_reference: digitize from Han et al. Fig 5 (non-locality → agreement = 1 − rate).
         Label as 'CaFE CLIP-L/14 (est. Fig 5)' with a dashed line.
"""
raise NotImplementedError("Person C — implement Spearman analysis (see docstring above)")

## 9. Layer evolution (Person C, D3)

Category distribution across layers 4 → 6 → 9 using CLIP labels from notebook 02.
Requires category annotations from D1 (`report/notes/feature_catalog_layer{4,6,9}.md`).
Produces **Fig 2**.

In [ ]:
"""Step 6 — layer evolution: concept category distribution across layers (Person C)."""
from src.visualise import plot_layer_evolution

feature_out_dir = repo_root / "outputs" / "features"
figures_dir     = repo_root / cfg.outputs.report_figures_path

def _classify_label(labels):
    texture_kw  = ["texture", "pattern", "surface", "grain", "downy", "feather", "lace", "bark", "lighting"]
    color_kw    = ["red", "blue", "green", "yellow", "orange", "purple", "pink", "white",
                   "black", "brown", "gray", "grey", "color", "colour", "hue", "bright", "dark"]
    part_kw     = ["beak", "wing", "neck", "collar", "leg", "eye", "tail", "tip", "claw"]
    scene_kw    = ["water", "sky", "grass", "background", "landscape", "shore", "marsh"]
    semantic_kw = ["bird", "animal", "dog", "cat", "face", "person", "vehicle", "insect", "plant"]
    text = " ".join(labels).lower()
    # check scene compound terms first, then color
    SCENE_COMPOUNDS = {"green grass", "blue sky", "green forest", "green field",
                    "green meadow", "green tree", "green vegetation",
                    "orange flower", "red flower", "flowing water", "water surface"}

    if any(k in text for k in part_kw):    return "part"
    if any(c in text for c in SCENE_COMPOUNDS): return "scene"   # ← new guard
    if any(k in text for k in texture_kw): return "texture"
    if any(k in text for k in color_kw):   return "color"
    if any(k in text for k in scene_kw):   return "scene"
    if any(k in text for k in semantic_kw):return "semantic"
    return "unclear"

category_counts = {}
for layer in cfg.sae.target_layers:
    labels_path = feature_out_dir / f"clip_labels_layer{layer}_full.json"
    if not labels_path.exists():
        print(f"[skip] layer {layer}: CLIP labels not found — run notebook 02 cell 4 first")
        continue
    with open(labels_path, encoding="utf-8") as f:
        feature_labels = {int(k): v for k, v in json.load(f).items()}

    counts = {"texture": 0, "color": 0, "part": 0, "scene": 0, "semantic": 0, "unclear": 0}
    for fi in selected_features_by_layer.get(layer, []):
        raw = feature_labels.get(fi, ["unknown"])
        counts[_classify_label(raw if isinstance(raw, list) else [raw])] += 1
    category_counts[layer] = counts
    print(f"Layer {layer}: {counts}")

if category_counts:
    save_path = str(figures_dir / "layer_evolution.png")
    fig = plot_layer_evolution(category_counts, save_path=save_path)
    display(fig)
    print(f"Saved → {save_path}")
else:
    print("No data — run notebook 02 CLIP labeling (cell 4) first.")

## Checkpoint

**Person C (cells 3a, 5, 6, 7a, 7b, 8, 9)**
- [ ] Run 1 CaFE IG — 100 features × 3 layers → `outputs/features/cafe_ig/run1/`
- [ ] Validate 5 features at layer 9 first: IG maps must be spatially coherent (non-uniform)
- [ ] Run 2 CaFE IG — 50 MS-ranked features × 3 layers → `outputs/features/cafe_ig/run2/`
- [ ] ERF visualisation figures saved to `report/figures/`
- [ ] CLIP Run 3 CaFE IG — 100 features × 3 layers → `outputs/features/cafe_ig/run3_clip/`
- [ ] Spearman results: DINO Run1 / Run2 / CLIP Run3 — per-layer ρ + pooled ρ
- [ ] **Fig 4** (locality by depth), **Fig 6 ★** (main comparison), **Fig 8** (MS scatter) saved
- [ ] **Fig 5** (locality by category) saved
- [ ] Layer evolution figure saved (`layer_evolution.png`)

**Person B (figure stubs + annotations)**
- [ ] `plot_locality_by_depth` implemented in `src/visualise.py` — **do this first** (unblocks Person C cell 8)
- [ ] `plot_locality_comparison` implemented → Fig 6 ★
- [ ] `plot_ms_locality_scatter` implemented → Fig 8
- [ ] `plot_locality_by_category` implemented → Fig 5
- [ ] Category annotations complete: `report/notes/feature_catalog_layer{4,6,9}.md`
- [ ] Category annotations complete: `report/notes/feature_catalog_clip_layer{4,6,9}.md`

**Figures not produced (dropped from plan):**
- MS distributions (Fig 1) → reported as table in report Methods section
- Category composition (Fig 2) → reported as table; `plot_layer_evolution` in nb03 cell 9 covers depth trend
- Insertion curves DINO/CLIP (Fig 3/7) → dropped; IG vs. AttnLRP limitation acknowledged in text